# Gold — Vendas em feriados vs dias normais

Este notebook cria a Gold de comparação entre vendas em feriados nacionais e dias normais.

Fontes:
- Silver `physical_vendas_caixa`
- Silver `feriados`

Granularidade:
- 1 linha por loja, ano, mês e tipo de dia

Destino ADLS:
- `gold/physical_vendas_caixa/vendas_feriados`

Destino SQL Server:
- `squad3.gold_physical_vendas_caixa_vendas_feriados`

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Gold.

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    current_timestamp,
    round as spark_round,
    sum as spark_sum,
    to_date,
    when
)

SILVER_VENDAS_TABLE = "physical_vendas_caixa"
SILVER_FERIADOS_TABLE = "feriados"

SILVER_VENDAS_PATH = f"{SILVER_BASE_PATH}{SILVER_VENDAS_TABLE}"
SILVER_FERIADOS_PATH = f"{SILVER_BASE_PATH}{SILVER_FERIADOS_TABLE}"

GOLD_DOMAIN = "physical_vendas_caixa"
GOLD_KPI = "vendas_feriados"

GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_DOMAIN}/{GOLD_KPI}"

FINAL_TABLE_NAME = f"gold_{GOLD_DOMAIN}_{GOLD_KPI}"
FINAL_TABLE = f"{TARGET_SCHEMA}.{FINAL_TABLE_NAME}"

GOLD_WRITE_MODE = "overwrite"

VENDAS_REQUIRED_COLUMNS = [
    "id_transacao",
    "id_loja",
    "dt_venda",
    "valor_total_venda",
    "ano",
    "mes"
]

FERIADOS_REQUIRED_COLUMNS = [
    "data_feriado",
    "nome_feriado",
    "tipo_feriado"
]

GOLD_KEY_COLUMNS = [
    "id_loja",
    "ano",
    "mes",
    "tipo_dia"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_VENDAS_PATH:", SILVER_VENDAS_PATH)
print("SILVER_FERIADOS_PATH:", SILVER_FERIADOS_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê as Silvers de vendas físicas e feriados.

df_vendas = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_VENDAS_PATH)
)

df_feriados = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_FERIADOS_PATH)
)

validate_required_columns(df_vendas, VENDAS_REQUIRED_COLUMNS)
validate_required_columns(df_feriados, FERIADOS_REQUIRED_COLUMNS)

total_vendas = df_vendas.count()
total_feriados = df_feriados.count()

print("Silvers lidas com sucesso.")
print(f"Total vendas físicas: {total_vendas}")
print(f"Total feriados: {total_feriados}")

print("Schema vendas:")
df_vendas.printSchema()

print("Schema feriados:")
df_feriados.printSchema()

display(df_vendas.limit(10))
display(df_feriados.limit(10))

In [0]:
# Prepara a data da venda e remove duplicidade de datas de feriado.

df_vendas_dia = (
    df_vendas
    .withColumn(
        "data_venda",
        to_date(col("dt_venda"))
    )
)

df_feriados_datas = (
    df_feriados
    .select(
        col("data_feriado")
    )
    .dropDuplicates(["data_feriado"])
)

print("Datas preparadas para o join.")
print(f"Total vendas: {df_vendas_dia.count()}")
print(f"Total datas distintas de feriado: {df_feriados_datas.count()}")

display(df_vendas_dia.select("id_transacao", "id_loja", "dt_venda", "data_venda").limit(10))
display(df_feriados_datas.orderBy("data_feriado").limit(10))

In [0]:
# Classifica cada venda como feriado ou dia normal.

df_vendas_classificadas = (
    df_vendas_dia
    .join(
        df_feriados_datas,
        df_vendas_dia["data_venda"] == df_feriados_datas["data_feriado"],
        "left"
    )
    .withColumn(
        "tipo_dia",
        when(col("data_feriado").isNotNull(), "Feriado").otherwise("Dia normal")
    )
    .drop("data_feriado")
)

print("Vendas classificadas por tipo de dia.")
print(f"Total após join: {df_vendas_classificadas.count()}")

display(
    df_vendas_classificadas
    .select(
        "id_transacao",
        "id_loja",
        "data_venda",
        "tipo_dia",
        "valor_total_venda"
    )
    .limit(30)
)

In [0]:
# Agrega vendas por loja, mês e tipo de dia.

df_vendas_feriados = (
    df_vendas_classificadas
    .groupBy(
        "id_loja",
        "ano",
        "mes",
        "tipo_dia"
    )
    .agg(
        count("id_transacao").alias("qtd_transacoes"),
        spark_sum("valor_total_venda").alias("receita_total"),
        countDistinct("data_venda").alias("qtd_dias_com_venda")
    )
    .withColumn(
        "ticket_medio",
        spark_round(col("receita_total") / col("qtd_transacoes"), 2)
    )
    .withColumn(
        "receita_media_por_dia",
        spark_round(col("receita_total") / col("qtd_dias_com_venda"), 2)
    )
    .withColumn(
        "transacoes_media_por_dia",
        spark_round(col("qtd_transacoes") / col("qtd_dias_com_venda"), 2)
    )
    .withColumn(
        "gold_processed_at",
        current_timestamp()
    )
)

print("Vendas em feriados vs dias normais calculadas.")
print(f"Total de linhas agregadas: {df_vendas_feriados.count()}")

display(
    df_vendas_feriados
    .orderBy("id_loja", "ano", "mes", "tipo_dia")
    .limit(40)
)

In [0]:
# Organiza o schema final da Gold.

df_gold_final = (
    df_vendas_feriados
    .select(
        "id_loja",
        "ano",
        "mes",
        "tipo_dia",
        "qtd_transacoes",
        "receita_total",
        "qtd_dias_com_venda",
        "ticket_medio",
        "receita_media_por_dia",
        "transacoes_media_por_dia",
        "gold_processed_at"
    )
)

print("Schema final da Gold definido.")

df_gold_final.printSchema()

display(
    df_gold_final
    .orderBy("id_loja", "ano", "mes", "tipo_dia")
    .limit(40)
)

In [0]:
# Valida volume, duplicidade e consistência da Gold.

total_gold = df_gold_final.count()

duplicados_gold = (
    df_gold_final
    .groupBy(GOLD_KEY_COLUMNS)
    .count()
    .filter(col("count") > 1)
    .count()
)

total_transacoes_gold = (
    df_gold_final
    .agg(spark_sum("qtd_transacoes").alias("qtd_transacoes"))
    .collect()[0]["qtd_transacoes"]
)

receita_silver = (
    df_vendas
    .agg(spark_sum("valor_total_venda").alias("receita_total"))
    .collect()[0]["receita_total"]
)

receita_gold = (
    df_gold_final
    .agg(spark_sum("receita_total").alias("receita_total"))
    .collect()[0]["receita_total"]
)

print(f"Total de linhas Gold: {total_gold}")
print(f"Chaves duplicadas na Gold: {duplicados_gold}")
print(f"Total transações Silver: {total_vendas}")
print(f"Total transações Gold: {total_transacoes_gold}")
print(f"Receita total Silver: {receita_silver}")
print(f"Receita total Gold: {receita_gold}")

if duplicados_gold > 0:
    raise Exception("Erro: existem chaves duplicadas na Gold.")

if total_transacoes_gold != total_vendas:
    raise Exception("Erro: total de transações da Gold não bate com a Silver.")

if receita_silver != receita_gold:
    raise Exception("Erro: receita total da Gold não bate com a Silver.")

print("Validação OK: Gold sem duplicidade e vendas consistentes.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold_final
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(GOLD_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(GOLD_PATH)
)

print(f"Gold Delta gravada com sucesso em: {GOLD_PATH}")

In [0]:
# Lê a Gold Delta gravada para validar persistência.

df_gold_delta = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

print("Gold Delta lida com sucesso.")
print(f"Total de linhas gravadas: {df_gold_delta.count()}")

df_gold_delta.printSchema()

display(
    df_gold_delta
    .orderBy("id_loja", "ano", "mes", "tipo_dia")
    .limit(40)
)

In [0]:
# Valida volume, duplicidade e consistência da Gold gravada.

total_gold_memoria = df_gold_final.count()
total_gold_delta = df_gold_delta.count()

duplicados_gold_delta = (
    df_gold_delta
    .groupBy(GOLD_KEY_COLUMNS)
    .count()
    .filter(col("count") > 1)
    .count()
)

total_transacoes_delta = (
    df_gold_delta
    .agg(spark_sum("qtd_transacoes").alias("qtd_transacoes"))
    .collect()[0]["qtd_transacoes"]
)

receita_gold_delta = (
    df_gold_delta
    .agg(spark_sum("receita_total").alias("receita_total"))
    .collect()[0]["receita_total"]
)

print(f"Total Gold em memória: {total_gold_memoria}")
print(f"Total Gold Delta: {total_gold_delta}")
print(f"Chaves duplicadas na Gold Delta: {duplicados_gold_delta}")
print(f"Total transações Silver: {total_vendas}")
print(f"Total transações Gold Delta: {total_transacoes_delta}")
print(f"Receita total Silver: {receita_silver}")
print(f"Receita total Gold Delta: {receita_gold_delta}")

if total_gold_memoria != total_gold_delta:
    raise Exception("Erro: quantidade gravada diferente da Gold em memória.")

if duplicados_gold_delta > 0:
    raise Exception("Erro: existem chaves duplicadas na Gold Delta.")

if total_transacoes_delta != total_vendas:
    raise Exception("Erro: total de transações da Gold Delta não bate com a Silver.")

if receita_silver != receita_gold_delta:
    raise Exception("Erro: receita total da Gold Delta não bate com a Silver.")

print("Validação OK: Gold Delta gravada corretamente.")

In [0]:
# Prepara a Gold para escrita no SQL Server.

df_gold_sql = (
    df_gold_delta
    .select(
        col("id_loja").cast("int").alias("id_loja"),
        col("ano").cast("int").alias("ano"),
        col("mes").cast("int").alias("mes"),
        col("tipo_dia").cast("string").alias("tipo_dia"),
        col("qtd_transacoes").cast("int").alias("qtd_transacoes"),
        col("receita_total").cast("decimal(18,2)").alias("receita_total"),
        col("qtd_dias_com_venda").cast("int").alias("qtd_dias_com_venda"),
        col("ticket_medio").cast("decimal(18,2)").alias("ticket_medio"),
        col("receita_media_por_dia").cast("decimal(18,2)").alias("receita_media_por_dia"),
        col("transacoes_media_por_dia").cast("decimal(18,2)").alias("transacoes_media_por_dia"),
        col("gold_processed_at").cast("timestamp").alias("gold_processed_at")
    )
)

print("Gold preparada para escrita no SQL Server.")

df_gold_sql.printSchema()

display(
    df_gold_sql
    .orderBy("id_loja", "ano", "mes", "tipo_dia")
    .limit(40)
)

In [0]:
# Grava a Gold diretamente na tabela final do SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print(f"Gold gravada com sucesso na tabela final: {FINAL_TABLE}")

In [0]:
# Lê e valida a tabela final gravada no SQL Server.

df_final = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

total_sql = df_final.count()

duplicados_sql = (
    df_final
    .groupBy(GOLD_KEY_COLUMNS)
    .count()
    .filter(col("count") > 1)
    .count()
)

total_transacoes_sql = (
    df_final
    .agg(spark_sum("qtd_transacoes").alias("qtd_transacoes"))
    .collect()[0]["qtd_transacoes"]
)

receita_sql = (
    df_final
    .agg(spark_sum("receita_total").alias("receita_total"))
    .collect()[0]["receita_total"]
)

print(f"Total Gold Delta: {total_gold_delta}")
print(f"Total tabela final SQL Server: {total_sql}")
print(f"Chaves duplicadas SQL Server: {duplicados_sql}")
print(f"Total transações SQL Server: {total_transacoes_sql}")
print(f"Receita Gold Delta: {receita_gold_delta}")
print(f"Receita SQL Server: {receita_sql}")

if total_sql != total_gold_delta:
    raise Exception("Erro: quantidade no SQL Server diferente da Gold Delta.")

if duplicados_sql > 0:
    raise Exception("Erro: existem chaves duplicadas no SQL Server.")

if total_transacoes_sql != total_vendas:
    raise Exception("Erro: total de transações no SQL Server não bate com a Silver.")

if receita_sql != receita_gold_delta:
    raise Exception("Erro: receita total no SQL Server não bate com a Gold Delta.")

print("Validação OK: tabela final SQL Server gravada corretamente.")